# SEED-0001 — R3 explicit seed-count sweep on fixed dev (A100)

This is a dev-only experiment. It generates one pool of 16 R3 candidates per question, each with its own deterministically derived seed, then evaluates plurality voting with the first 1, 4, 8, 12, and 16 candidates. It does not read the leaderboard or final test, and it does not run PAL or adaptive length.

In [ ]:
# Cell 1 — Fresh A100 only. Run once, then restart the runtime before Cell 2.
%pip install -q --no-cache-dir "nvidia-cuda-runtime==13.0.88" "nvidia-cuda-nvrtc==13.0.88" "vllm==0.26.0" "pandas>=2.2,<3"
print("[SETUP] Restart the runtime once, then run Cells 2–5 in order.")


In [ ]:
# Cell 2 — Mount Drive, update the private repository, and cache the pinned base model.
from google.colab import drive, userdata
from pathlib import Path
import subprocess

drive.mount('/content/drive')
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = 'https://github.com/jhparktime/qwen-math-final-2026.git'
if token:
    url = 'https://x-access-token:' + token + '@github.com/jhparktime/qwen-math-final-2026.git'
repo = Path('/content/qwen-math-final')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', url, str(repo)], check=True)
%cd /content/qwen-math-final
!python3 scripts/prefetch_model.py


In [ ]:
# Cell 3 — Freeze the held dev split and R3 adapter.
import re, unicodedata
from pathlib import Path

def compact(value):
    return re.sub(r'[\s_-]+', '', unicodedata.normalize('NFC', str(value)).casefold())

roots = [p for p in Path('/content/drive/MyDrive').iterdir() if p.is_dir() and compact(p.name) == compact('2026소중한챌린지')]
assert len(roots) == 1, roots
project = roots[0]
runs = project / 'runs'
SPLIT_DIR = runs / 'AUDIT-0002-clean-split-passN-20260821-215706' / 'splits'
DEV_PATH = SPLIT_DIR / 'dev_v1.csv'
R3_ADAPTER = runs / 'RFT-0008D-r3mix-r2continue-r16-a100' / 'adapter_final'
OUTPUT_DIR = runs / 'SEED-0001-r3-explicit-seedcount-dev'
for path in [DEV_PATH, R3_ADAPTER / 'adapter_config.json', R3_ADAPTER / 'adapter_model.safetensors']:
    assert path.exists(), path
print({'dev': str(DEV_PATH), 'r3_adapter': str(R3_ADAPTER), 'output': str(OUTPUT_DIR), 'base_seed': 3, 'seed_counts': [1, 4, 8, 12, 16]})


In [ ]:
# Cell 4 — Generate one explicit 16-seed pool. Resume-safe.
import os, subprocess, sys
env = dict(os.environ, PYTHONPATH='.')
subprocess.run([sys.executable, 'inference/seed_diversity_dev.py', '--input', str(DEV_PATH), '--adapter', str(R3_ADAPTER), '--output-dir', str(OUTPUT_DIR), '--base-seed', '3'], check=True, env=env)


In [ ]:
# Cell 4B — Score any complete existing pool without loading vLLM or using the GPU.
RAW_POOL = OUTPUT_DIR / 'candidates' / 'dev_r3_explicit16_seed3_pool.jsonl'
subprocess.run([sys.executable, 'inference/score_seed_diversity_dev.py', '--input', str(DEV_PATH), '--raw', str(RAW_POOL), '--output-dir', str(OUTPUT_DIR)], check=True, env=env)


In [ ]:
# Cell 5 — Read the exact dev sweep. Do not promote a count based on this alone without retaining this report.
import json, pandas as pd
report = json.loads((OUTPUT_DIR / 'reports' / 'experiment_report.json').read_text())
sweep = pd.read_csv(OUTPUT_DIR / 'reports' / 'seed_count_sweep.csv')
display(sweep[['seed_count', 'correct', 'exact_match', 'tie_rows', 'parse_failure_candidates', 'mean_unique_answers']])
print(json.dumps({'seed16_vs_seed1': report['seed16_vs_seed1'], 'raw_pool': report['artifacts']['raw_pool'], 'report': str(OUTPUT_DIR / 'reports' / 'experiment_report.json')}, ensure_ascii=False, indent=2))


In [ ]:
# Final cell — optional GPU release after artifacts are saved.
DISCONNECT_GPU_RUNTIME = False
if DISCONNECT_GPU_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print('[RUNTIME] retained. Set DISCONNECT_GPU_RUNTIME=True when finished.')
